In [1]:
import nbimporter

import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd
import seaborn as sns
from pyproj import Proj
from mod13q1_preprocessing import process_mod13q1
from mod11a2_preprocessing import process_combined_mod11a2

**Align MOD13Q1 and MOD11A2 datasets**

In [2]:
def align_datasets(mod13q1_file, mod11a2_files, mod11a2_path, target_tile):
    """
    Align MOD13Q1 (NDVI & EVI) with MOD11A2 (LST) spatially.
    All outputs are (1200, 1200) NumPy arrays.
    
    Returns:
        ndvi_1km, evi_1km, lst_day_1km, lst_night_1km
    """

    print("..target_tile ", target_tile)
    ndvi_1km, evi_1km = process_mod13q1(mod13q1_file)
    lst_day_1km, lst_night_1km = process_combined_mod11a2(mod11a2_files, mod11a2_path, target_tile)

    print(f"After processing:")
    print(f"MOD13Q1 NDVI aggregated shape: {ndvi_1km.shape}")
    print(f"MOD13Q1 EVI aggregated shape: {evi_1km.shape}")
    print(f"MOD11A2 LST Day shape: {lst_day_1km.shape}")
    print(f"MOD11A2 LST Night shape: {lst_night_1km.shape}")

    return ndvi_1km, evi_1km, lst_day_1km, lst_night_1km

**Stack Features & Mask Invalid Pixels**

In [3]:
def create_features(ndvi, evi, lst_day, lst_night):
    """
    Combine NDVI, EVI, LST Day, and LST Night into a single (1200, 1200, 4) feature array.
    Also return a mask for valid (non-NaN) pixels across all features.
    
    Returns:
        features: stacked feature array with shape (1200, 1200, 4)
        valid_mask: boolean mask where True indicates all features are valid (non-NaN)
    """

    features = np.stack([ndvi, evi, lst_day, lst_night], axis=-1)  # shape (1200, 1200, 4)
    
    # Mask out pixels where any feature is NaN
    valid_mask = ~np.any(np.isnan(features), axis=-1)
    
    return features, valid_mask

**Projection and FIRMS matching**

In [4]:
def latlon_to_tile_pixel(lat, lon):
    """
    Convert latitude/longitude to MODIS tile indices and 1km pixel coordinates within tile.
    
    Returns:
        h (int): MODIS tile horizontal index
        v (int): MODIS tile vertical index
        i (int): column in tile (0–1199)
        j (int): row in tile (0–1199)
    """
    # Define MODIS Sinusoidal projection (used to convert lat/lon to x/y)
    modis_sinu_proj = Proj('+proj=sinu +R=6371007.181 +nadgrids=@null +wktext')
    
    # Convert lat/lon to sinusoidal x/y
    x, y = modis_sinu_proj(lon, lat)

    # Accurate constants from NASA MODIS grid
    PIXEL_SIZE_M = 926.62543305         # More precise 1km pixel resolution in meters
    TILE_SIZE_M = PIXEL_SIZE_M * 1200   # = 1111950 * 10 = ~1200km per tile
    ORIGIN_X = -20015109.354            # MODIS Sinusoidal grid X origin (left)
    ORIGIN_Y = 10007554.677             # MODIS Sinusoidal grid Y origin (top)

    # Convert lat/lon to sinusoidal projection
    x, y = modis_sinu_proj(lon, lat)

    # Compute tile indices (horizontal h, vertical v)
    h = int((x - ORIGIN_X) // TILE_SIZE_M)
    v = int((ORIGIN_Y - y) // TILE_SIZE_M)

    # Compute pixel indices within the tile
    i = int(((x - ORIGIN_X) % TILE_SIZE_M) // PIXEL_SIZE_M)
    j = int(((ORIGIN_Y - y) % TILE_SIZE_M) // PIXEL_SIZE_M)

    # Ensure within tile bounds
    if not (0 <= i < 1200 and 0 <= j < 1200):
        return None  # Point is outside valid tile range

    return h, v, i, j

**Perform exploratory data analysis (EDA) on input features and fire labels.**

In [5]:
def run_feature_eda(X_all, y_all):
    """
    Perform exploratory data analysis (EDA) on input features and fire labels.
    
    Parameters:
        X_all (array-like): Feature matrix with shape (n_samples, 4)
        y_all (array-like): Corresponding binary fire labels (0 or 1)
    """
    if X_all is None or len(X_all) == 0:
        print("No data to analyze.")
        return

    # Create DataFrame
    feature_names = ['NDVI', 'EVI', 'LST_Day', 'LST_Night']
    df = pd.DataFrame(X_all, columns=feature_names)
    df['fire_label'] = y_all

    # 1. Histograms of feature distributions
    plt.figure(figsize=(16, 10))
    for i, col in enumerate(feature_names):
        plt.subplot(2, 2, i + 1)
        sns.histplot(df[col], bins=50, kde=True, color='blue')
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    # 2. Feature distributions by fire label
    custom_palette = {0: 'blue', 1: 'red'}
    plt.figure(figsize=(16, 10))
    for i, col in enumerate(feature_names):
        plt.subplot(2, 2, i + 1)
        sns.histplot(
            data=df, 
            x=col, 
            hue='fire_label', 
            bins=50, 
            kde=True, 
            element="step", 
            stat="density", 
            common_norm=False,
            palette=custom_palette
        )
        plt.title(f'{col} by Fire vs Non-Fire')
        plt.xlabel(col)
        plt.ylabel('Density')
    plt.tight_layout()
    plt.show()

    # 3. Correlation heatmap
    corr_matrix = df.corr(method='pearson')
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", square=True)
    plt.title('Correlation Matrix (including Fire Label)')
    plt.tight_layout()
    plt.show()

    # 4. Mean difference between fire and non-fire samples
    fire_means = df[df['fire_label'] == 1][feature_names].mean()
    nonfire_means = df[df['fire_label'] == 0][feature_names].mean()
    diff = fire_means - nonfire_means
    print("Mean Difference (Fire - Non-Fire):\n", diff)